In [4]:
import json, csv, pickle, os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score



# ── Load data ─────────────────────────────────────────────────────────────────
rows = list(csv.DictReader(open("pipeline_dataset.csv", encoding="utf-8")))
texts  = [" ".join(json.loads(r["tokens_stemmed"])) for r in rows]
labels = [r["label"] for r in rows]

LABEL_ORDER = ["low", "medium", "high"]
label2int   = {l: i for i, l in enumerate(LABEL_ORDER)}
int2label   = {i: l for l, i in label2int.items()}

# ── Eval on 80/20 split for honest metrics ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)
svm_eval = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ("svm",   LinearSVC(C=1.0, max_iter=2000, random_state=42)),
])
svm_eval.fit(X_train, y_train)
preds    = svm_eval.predict(X_test)
eval_acc = accuracy_score(y_test, preds)
eval_f1  = f1_score(y_test, preds, average="weighted")

# ── Train final model on FULL dataset ────────────────────────────────────────
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ("svm",   LinearSVC(C=1.0, max_iter=2000, random_state=42)),
])
svm_pipeline.fit(texts, labels)

# ── Save model ────────────────────────────────────────────────────────────────
model_path ="svm_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(svm_pipeline, f)
print(f"✅  svm_model.pkl saved  ({os.path.getsize(model_path):,} bytes)")

# ── Save metadata ─────────────────────────────────────────────────────────────
meta = {
    "model":            "LinearSVC",
    "feature":          "TF-IDF (unigram+bigram, min_df=2, sublinear_tf=True)",
    "input_field":      "tokens_stemmed joined with spaces",
    "classes":          LABEL_ORDER,
    "label2int":        label2int,
    "int2label":        {str(k): v for k, v in int2label.items()},
    "trained_on":       len(texts),
    "eval_accuracy":    round(eval_acc, 4),
    "eval_weighted_f1": round(eval_f1, 4),
    "how_to_load": (
        "import pickle\n"
        "model = pickle.load(open('svm_model.pkl', 'rb'))\n"
        "model.predict(['your normalized stemmed text here'])"
    ),
}
meta_path ="label_meta.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"✅  label_meta.json saved  ({os.path.getsize(meta_path):,} bytes)")

# ── Smoke test: reload from disk and predict ──────────────────────────────────
print("\n── Smoke test ──")
loaded_model = pickle.load(open(model_path, "rb"))

test_cases = [
    ("रगत बगएकओ बएहओस तक",        "high"),
    ("हल्का बान्ता कम दुखाइ",       "medium"),
    ("दिनभरि पानि सब ठिक सहज",     "low"),
]
all_ok = True
for text, expected in test_cases:
    pred = loaded_model.predict([text])[0]
    ok   = pred == expected
    if not ok: all_ok = False
    print(f"  {'✅' if ok else '❌'}  pred={pred:6s}  expected={expected:6s}  | {text}")

print()
print("All smoke tests passed ✅" if all_ok else "Some smoke tests FAILED ❌")
print(f"\nEval accuracy   : {eval_acc*100:.2f}%")
print(f"Eval weighted F1: {eval_f1*100:.2f}%")

✅  svm_model.pkl saved  (47,723 bytes)
✅  label_meta.json saved  (584 bytes)

── Smoke test ──
  ✅  pred=high    expected=high    | रगत बगएकओ बएहओस तक
  ✅  pred=medium  expected=medium  | हल्का बान्ता कम दुखाइ
  ✅  pred=low     expected=low     | दिनभरि पानि सब ठिक सहज

All smoke tests passed ✅

Eval accuracy   : 74.47%
Eval weighted F1: 74.42%
